In [ ]:
#!pip install pandas numpy matplotlib seaborn scikit-learn statsmodels openpyxl

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [2]:
# Load the data
excel_file = 'SOBSS Branded PS Data 2023.xlsx'

# Inspect available sheets
xls = pd.ExcelFile(excel_file)
print("Available sheets:")
print(xls.sheet_names)
print("\n")

# Read all data sheets (exclude the Data Dictionary)
data_sheets = [s for s in xls.sheet_names if s.lower().strip() != 'data dictionary']
print(f"Loading sheets: {data_sheets}")

df_list = []
for s in data_sheets:
    df_tmp = pd.read_excel(excel_file, sheet_name=s)
    df_tmp['source_sheet'] = s
    df_list.append(df_tmp)

# Concatenate all region/channel sheets
df_raw = pd.concat(df_list, ignore_index=True)


print(f"Combined raw shape: {df_raw.shape}")



Available sheets:
['Data Dictionary', 'US SEO', 'US PPC', 'Canada PPC', 'Canada SEO']


Loading sheets: ['US SEO', 'US PPC', 'Canada PPC', 'Canada SEO']
Combined raw shape: (13365, 14)


In [3]:
# Convert day of month to day of week (categorical: Monday, Tuesday, etc.)
# Create a temporary datetime column to extract day of week
df_raw['temp_date'] = pd.to_datetime(df_raw[['year', 'month', 'day']].astype(str).agg('-'.join, axis=1))
df_raw['day_of_week'] = df_raw['temp_date'].dt.day_name()

# Convert to categorical for better analysis
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_raw['day_of_week'] = pd.Categorical(df_raw['day_of_week'], categories=day_order, ordered=True)

# Drop the temporary date column
df_raw = df_raw.drop(columns=['temp_date'])

print('\nDay of week conversion complete:')
print(f"Day of week categories: {df_raw['day_of_week'].cat.categories.tolist()}")
print(f"Unique day_of_week values: {sorted(df_raw['day_of_week'].unique())}")




Day of week conversion complete:
Day of week categories: ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
Unique day_of_week values: ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']


In [4]:
# Create dummy variables for day of week (6 dummies, Sunday as reference)
day_dummies = pd.get_dummies(df_raw['day_of_week'], prefix='day', drop_first=True)

# Convert boolean dummies to int for regression
day_dummies = day_dummies.astype(int)

# Add dummy columns to df_raw
df_raw = pd.concat([df_raw, day_dummies], axis=1)

print('\nDay of week dummy variables created:')
print(f"Dummy columns: {day_dummies.columns.tolist()}")
print(f"Reference category: Sunday (omitted to avoid multicollinearity)")
print(f"Shape after adding dummies: {df_raw.shape}")




Day of week dummy variables created:
Dummy columns: ['day_Tuesday', 'day_Wednesday', 'day_Thursday', 'day_Friday', 'day_Saturday', 'day_Sunday']
Reference category: Sunday (omitted to avoid multicollinearity)
Shape after adding dummies: (13365, 21)


In [5]:
# Normalize some column names for easier handling
rename_map = {
    'BRAND_ads_ON_Flag (valid during test period only)': 'BRAND_ads_ON_Flag',
    'Hour Designation (during test period)': 'Hour_Designation',
    'Test Preiod (1 = Test Period, 0 = Non-test period)': 'Test_Period'
}

df_raw = df_raw.rename(columns=rename_map)

df_raw = df_raw.drop(columns= ['Hour_Designation','cnt_obsrv'])

print('\nColumns after rename:')
print(df_raw.columns.tolist())




Columns after rename:
['Country', 'Channel', 'year', 'month', 'day', 'Hour', 'Test_Period', 'BRAND_ads_ON_Flag', 'Users', 'NewUsers', 'Trials', 'source_sheet', 'day_of_week', 'day_Tuesday', 'day_Wednesday', 'day_Thursday', 'day_Friday', 'day_Saturday', 'day_Sunday']


In [6]:
# Aggregation keys (per country totals) - now using day of week instead of day of month
agg_keys = ['year', 'month', 'day_of_week', 'Hour', 'BRAND_ads_ON_Flag']
for k in agg_keys:
    if k not in df_raw.columns:
        raise KeyError(f"Missing aggregation key: {k}")

# Total for United States
df_total_US = (
    df_raw[df_raw['Country'].str.lower().str.contains('united')]
    .groupby(agg_keys)[['Users', 'NewUsers', 'Trials', 'day_Tuesday', 'day_Wednesday', 'day_Thursday', 'day_Friday', 'day_Saturday', 'day_Sunday']]
    .sum()
    .reset_index()
    .rename(columns={'Users': 'total_users_US', 'NewUsers': 'total_newusers_US', 'Trials': 'total_trials_US'})
)

# Total for Canada
df_total_Canada = (
    df_raw[df_raw['Country'].str.lower().str.contains('canada')]
    .groupby(agg_keys)[['Users', 'NewUsers', 'Trials', 'day_Tuesday', 'day_Wednesday', 'day_Thursday', 'day_Friday', 'day_Saturday', 'day_Sunday']]
    .sum()
    .reset_index()
    .rename(columns={'Users': 'total_users_CA', 'NewUsers': 'total_newusers_CA', 'Trials': 'total_trials_CA'})
)

print(f"\nTotal US aggregated shape: {df_total_US.shape}")
print(f"Total Canada aggregated shape: {df_total_Canada.shape}")

# Show a few rows of each aggregated DF
print('\nSample df_total_US:')
print(df_total_US.head())
print('\nSample df_total_Canada:')
print(df_total_Canada.head())





Total US aggregated shape: (1680, 14)
Total Canada aggregated shape: (1680, 14)

Sample df_total_US:
   year  month day_of_week  Hour  BRAND_ads_ON_Flag  total_users_US  \
0  2023      5      Monday     0                  0             106   
1  2023      5      Monday     0                  1               0   
2  2023      5      Monday     1                  0              92   
3  2023      5      Monday     1                  1               0   
4  2023      5      Monday     2                  0               0   

   total_newusers_US  total_trials_US  day_Tuesday  day_Wednesday  \
0                 42                0            0              0   
1                  0                0            0              0   
2                 25                2            0              0   
3                  0                0            0              0   
4                  0                0            0              0   

   day_Thursday  day_Friday  day_Saturday  day_Sunday  


In [7]:
# Filter out rows where BRAND_ads_ON_Flag = 1 but all outcome variables are 0
# These represent treatment periods with no data

print("Before filtering:")
print(f"US total shape: {df_total_US.shape}")
print(f"Canada total shape: {df_total_Canada.shape}")

# For US total
df_total_US = df_total_US[~(
    (df_total_US['BRAND_ads_ON_Flag'] == 1) & 
    (df_total_US['total_users_US'] == 0) & 
    (df_total_US['total_newusers_US'] == 0) & 
    (df_total_US['total_trials_US'] == 0)
)]

# For Canada total
df_total_Canada = df_total_Canada[~(
    (df_total_Canada['BRAND_ads_ON_Flag'] == 1) & 
    (df_total_Canada['total_users_CA'] == 0) & 
    (df_total_Canada['total_newusers_CA'] == 0) & 
    (df_total_Canada['total_trials_CA'] == 0)
)]

print("After filtering:")
print(f"US total shape: {df_total_US.shape}")
print(f"Canada total shape: {df_total_Canada.shape}")

Before filtering:
US total shape: (1680, 14)
Canada total shape: (1680, 14)
After filtering:
US total shape: (1260, 14)
Canada total shape: (1260, 14)


In [8]:
df_total_Canada.head()

,year,month,day_of_week,Hour,BRAND_ads_ON_Flag,total_users_CA,total_newusers_CA,total_trials_CA,day_Tuesday,day_Wednesday,day_Thursday,day_Friday,day_Saturday,day_Sunday
0,2023,5,Monday,0,0,57,22,3,0,0,0,0,0,0
2,2023,5,Monday,1,0,37,9,4,0,0,0,0,0,0
4,2023,5,Monday,2,0,0,0,0,0,0,0,0,0,0
5,2023,5,Monday,2,1,18,6,2,0,0,0,0,0,0
6,2023,5,Monday,3,0,0,0,0,0,0,0,0,0,0


In [9]:
df_total_US.head()

,year,month,day_of_week,Hour,BRAND_ads_ON_Flag,total_users_US,total_newusers_US,total_trials_US,day_Tuesday,day_Wednesday,day_Thursday,day_Friday,day_Saturday,day_Sunday
0,2023,5,Monday,0,0,106,42,0,0,0,0,0,0,0
2,2023,5,Monday,1,0,92,25,2,0,0,0,0,0,0
4,2023,5,Monday,2,0,0,0,0,0,0,0,0,0,0
5,2023,5,Monday,2,1,54,20,1,0,0,0,0,0,0
6,2023,5,Monday,3,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Create 4 raw dataframes for each country and channel combination
# US SEO
df_us_seo = df_raw[(df_raw['Country'].str.lower().str.contains('united')) & (df_raw['source_sheet'] == 'US SEO')].copy()

# US PPC
df_us_ppc = df_raw[(df_raw['Country'].str.lower().str.contains('united')) & (df_raw['source_sheet'] == 'US PPC')].copy()

# Canada SEO
df_ca_seo = df_raw[(df_raw['Country'].str.lower().str.contains('canada')) & (df_raw['source_sheet'] == 'Canada SEO')].copy()

# Canada PPC
df_ca_ppc = df_raw[(df_raw['Country'].str.lower().str.contains('canada')) & (df_raw['source_sheet'] == 'Canada PPC')].copy()

print(f"US SEO shape: {df_us_seo.shape}")
print(f"US PPC shape: {df_us_ppc.shape}")
print(f"Canada SEO shape: {df_ca_seo.shape}")
print(f"Canada PPC shape: {df_ca_ppc.shape}")


US SEO shape: (3453, 19)
US PPC shape: (3410, 19)
Canada SEO shape: (3345, 19)
Canada PPC shape: (3157, 19)

US SEO sample:
         Country Channel  year  month  day  Hour  Test_Period  \
0  United States     SEO  2023      5    1     0            0   
1  United States     SEO  2023      5    1     1            0   
2  United States     SEO  2023      5    1     2            0   

   BRAND_ads_ON_Flag  Users  NewUsers  Trials source_sheet day_of_week  \
0                  0     12         3       0       US SEO      Monday   
1                  0     12         4       0       US SEO      Monday   
2                  1      9         1       0       US SEO      Monday   

   day_Tuesday  day_Wednesday  day_Thursday  day_Friday  day_Saturday  \
0            0              0             0           0             0   
1            0              0             0           0             0   
2            0              0             0           0             0   

   day_Sunday  
0        

In [12]:
df_us_ppc.head()

,Country,Channel,year,month,day,Hour,Test_Period,BRAND_ads_ON_Flag,Users,NewUsers,Trials,source_sheet,day_of_week,day_Tuesday,day_Wednesday,day_Thursday,day_Friday,day_Saturday,day_Sunday
3453,United States,PPC Branded,2023,5,1,0,0,0,9,4,0,US PPC,Monday,0,0,0,0,0,0
3454,United States,PPC Branded,2023,5,1,1,0,0,8,4,1,US PPC,Monday,0,0,0,0,0,0
3455,United States,PPC Branded,2023,5,1,2,0,1,8,7,1,US PPC,Monday,0,0,0,0,0,0
3456,United States,PPC Branded,2023,5,1,3,0,1,5,3,0,US PPC,Monday,0,0,0,0,0,0
3457,United States,PPC Branded,2023,5,1,4,0,0,1,0,0,US PPC,Monday,0,0,0,0,0,0
